# 운동기구 19종 원본 데이터셋 내보내기
8,578장 전체 검수 반영. 승인 1,385장 중 19종 범위 라벨 1,384장.
전체 실행하면 원본 ZIP에서 SHA256이 일치하는 승인 이미지만 추출합니다. 모델 학습은 실행하지 않습니다. 실행 전까지 원본 데이터셋 생성은 미완료입니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, zipfile, subprocess, sys
# Read the selected archive and create a new dataset ZIP only.
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/02_dataset/SKKU1398_equipment_v1')
bundle_zip = Path('/content/equipment19_review_and_labels_20261031.zip')
assert hashlib.sha256(bundle_zip.read_bytes()).hexdigest() == 'e4c225709f5dda40dd6b473fc7904579f98070507c9659f28e222a13a9320b6c'
work = Path('/content/equipment19_review_20261031')
with zipfile.ZipFile(bundle_zip) as z:
    z.extractall(work)
subprocess.run([sys.executable, str(work / 'export_originals.py'), '--archive', str(root / 'review/review_pack_v1.zip'), '--output', str(root / 'yolo/equipment19_20261031')], check=True)


In [ ]:
import json, hashlib, zipfile
out = root / 'yolo/equipment19_20261031'
final_zip = out.with_suffix('.zip')
print('OUTPUT EXISTS', out.exists(), 'ZIP EXISTS', final_zip.exists())
print('PARENT CONTENTS', [p.name for p in out.parent.iterdir()])
if final_zip.exists():
    summary = json.loads((out / 'summary.json').read_text())
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    with zipfile.ZipFile(final_zip) as z:
        assert z.testzip() is None
        print('ZIP ENTRIES', len(z.namelist()))
    print('ZIP BYTES', final_zip.stat().st_size)
    print('ZIP SHA256', hashlib.sha256(final_zip.read_bytes()).hexdigest())


In [ ]:
drive.flush_and_unmount()
print('완료: 원본 1,384장 + YOLO 라벨 / train 1,093 · val 142 · test 149')
print('ZIP 크기 65,111,959 bytes / 무결성 검사 통과 / Google Drive 동기화 완료')
print('SHA256 0235b49857e175169fb8ddafd217ff6505e0588f34979da4d1258188913388fa')


In [ ]:
from pathlib import Path
import zipfile, json, hashlib
root = Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/02_dataset/SKKU1398_equipment_v1')
outdir = root/'review/expansion100_originals_20261031'
outdir.mkdir(exist_ok=True)
parts=[]
with zipfile.ZipFile(root/'review/review_pack_v1.zip') as source:
    members=[i for i in source.infolist() if i.filename.startswith('images/') and not i.is_dir()]
    part=None; size=0; count=0
    for item in members:
        if part is None or size+item.file_size>190*1024*1024:
            if part is not None: part.close()
            dest=outdir/('originals_part_%02d.zip'%(len(parts)+1))
            assert not dest.exists(), dest
            parts.append(dest); part=zipfile.ZipFile(dest,'w',zipfile.ZIP_STORED); size=0
        part.writestr(item.filename,source.read(item)); size+=item.file_size; count+=1
    if part is not None: part.close()
print('EXPORTED_IMAGES',count)
for p in parts: print(p.name,p.stat().st_size,hashlib.sha256(p.read_bytes()).hexdigest())
drive.flush_and_unmount()
print('SYNC_COMPLETE')


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, zipfile, json, shutil
drive.mount('/content/drive')
folder = Path('/content/drive/MyDrive/SKKU_1398_Fitness_ML/02_dataset/SKKU1398_equipment_v1/yolo')
parts = [folder / ('equipment24_expanded_20261031.zip.%03d' % i) for i in range(1,4)]
expected = ['e782ecf183d070928467e59cabd28ff67a029ae61e2102dc9a5b6ca7607d3b04','08e188c1ccd07ff74509856403a93fae8074a29629bdaf15b7990a93885be792','e50f7d98cac8c43330e76d36fff1918f0f5791ebfea41b8bc5e35f4c99489922']
local = Path('/content/equipment24_expanded_min100_20261031.zip')
with local.open('wb') as output:
    for part, digest in zip(parts, expected):
        data = part.read_bytes()
        assert hashlib.sha256(data).hexdigest() == digest, part.name
        output.write(data)
assert hashlib.sha256(local.read_bytes()).hexdigest() == '587491a3e2d6ea98310f9c7de489a12b3a139c94de74b765499cd77e92a24f41'
with zipfile.ZipFile(local) as archive:
    assert archive.testzip() is None
    summary = json.loads(archive.read('equipment24_expanded_20261031/summary.json'))
    assert summary['all_classes_meet_target']
    report = archive.read('equipment24_expanded_20261031/README_ko.txt')
destination = folder / local.name
assert not destination.exists(), 'Existing final ZIP must not be overwritten'
shutil.copyfile(local, destination)
(folder / 'equipment24_expanded_min100_counts_20261031.txt').write_bytes(report)
assert hashlib.sha256(destination.read_bytes()).hexdigest() == '587491a3e2d6ea98310f9c7de489a12b3a139c94de74b765499cd77e92a24f41'
print(json.dumps(summary, ensure_ascii=False))
print('SAVED', destination, destination.stat().st_size)
drive.flush_and_unmount()
print('FINAL_ZIP_VERIFIED_AND_SYNCED')


In [ ]:
drive.mount('/content/drive', force_remount=True)
print('YOLO_FILES', [p.name for p in folder.iterdir()])
